In [140]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from ollama import chat
import os
from ragas import evaluate
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from ragas.llms import llm_factory
from ragas.metrics.collections import faithfulness, answer_relevancy, context_recall
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.cache import DiskCacheBackend
from ragas.embeddings import embedding_factory
import pandas as pd
from datasets import Dataset

from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

Настройка API ключей

In [141]:
load_dotenv()
HF_API_KEY = os.getenv("HF_API_KEY")
os.environ["HF_TOKEN"] = HF_API_KEY

In [142]:
file_path = os.getenv("PDF_FILE_PATH")
loader = PyPDFLoader(file_path)
docs = loader.load()

Инициализация эмбеддингов

In [143]:
hf_embeddings_model = HuggingFaceEmbeddings(
    model_name="cointegrated/LaBSE-en-ru",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22785.17it/s]
BertModel LOAD REPORT from: cointegrated/LaBSE-en-ru
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Чанки и Сплиттер

In [ ]:
def split_markdown_by_separator_for_rag(
    file_path: str,
    separator: str = "—————",
    chunk_size: int = 500,
    chunk_overlap: int = 100,
):
    text = Path(file_path).read_text(encoding="utf-8")

    # 1. делим на смысловые блоки по разделителю
    raw_blocks = [block.strip() for block in text.split(separator)]
    raw_blocks = [block for block in raw_blocks if block]
    # 2. создаем документы по блокам
    block_docs = [
        Document(
            page_content=block,
            metadata={"block_id": i + 1}
        )
        for i, block in enumerate(raw_blocks)
    ]

    # 3. режем блок на чанки
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
    )

    final_chunks = []
    for doc in block_docs:
        chunks = splitter.create_documents(
            [doc.page_content],
            metadatas=[doc.metadata]
        )

        # номер чанка внутри блока
        for j, chunk in enumerate(chunks, start=1):
            chunk.metadata["chunk_in_block"] = j
            chunk.metadata["total_chunks_in_block"] = len(chunks)

        final_chunks.extend(chunks)

    return final_chunks

# Использование
chunk_documents = split_markdown_by_separator_for_rag(
    file_path="RAG.md",
    separator="—————",
    chunk_size=1000,
    chunk_overlap=100,
)

print(f"Всего чанков: {len(chunk_documents)}")


Всего чанков: 143


Создание или загрузка векторной базы Chroma

In [145]:
import os
import shutil
import tempfile

# тут временная папка с гарантированными правами
persist_directory = tempfile.mkdtemp()

print(f"папка для базы: {persist_directory}")

vector_db = Chroma.from_documents(
    documents=chunk_documents,
    embedding=hf_embeddings_model,
    persist_directory=persist_directory
)

папка для базы: /tmp/tmprecrj0il


Настройка ретриверов и поиск контекста

In [146]:
my_text = "Разрешено ли находиться в вузе ночью?"

vector_retriever = vector_db.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(chunk_documents)
bm25_retriever.k = 10

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.0, 1]
)

retriever_results = ensemble_retriever.invoke(my_text)
print(retriever_results)
print()

[Document(id='d6bd1290-1e3b-42aa-82ad-e73f4860359e', metadata={'total_chunks_in_block': 1, 'block_id': 66, 'chunk_in_block': 1}, page_content='## Кто может жить в общежитии?\n\n• Иногородних студентов (бакалавриат, специалитет, магистратура, аспирантура).\n• Иностранных студентов, включая участников программ Минобрнауки.\n• Абитуриентов на период вступительных экзаменов. \n• Сотрудников университета на время работы.\n\nВажно: Семейным парам и студентам с детьми места не предоставляются.'), Document(id='0ee20ef1-10dc-4b32-ad7c-2f93e2809b85', metadata={'block_id': 2, 'total_chunks_in_block': 1, 'chunk_in_block': 1}, page_content='## О курении и алкоголе\n\n\nНа территории университета и в общежитии курение и распитие алкогольных\nнапитков, а также нахождение в нетрезвом состоянии СТРОГО ЗАПРЕЩЕНО!'), Document(id='40076377-c36a-4ccf-bafc-d10074812534', metadata={'block_id': 7, 'chunk_in_block': 2, 'total_chunks_in_block': 3}, page_content='Стратегическая цель института — это подготовка ка

Реранкер и формирование контекста из найденных чанков

In [147]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)

pairs = [[my_text, doc.page_content] for doc in retriever_results]
rerank_scores = reranker.predict(pairs)

sorted_indices = sorted(range(len(rerank_scores)), key=lambda i: rerank_scores[i], reverse=True)
top_k = 5
reranked_docs = [retriever_results[i] for i in sorted_indices[:top_k]]

context = "\n\n".join([doc.page_content for doc in reranked_docs])
print(context)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 21625.80it/s]


## О курении и алкоголе


На территории университета и в общежитии курение и распитие алкогольных
напитков, а также нахождение в нетрезвом состоянии СТРОГО ЗАПРЕЩЕНО!

## Кто может жить в общежитии?

• Иногородних студентов (бакалавриат, специалитет, магистратура, аспирантура).
• Иностранных студентов, включая участников программ Минобрнауки.
• Абитуриентов на период вступительных экзаменов. 
• Сотрудников университета на время работы.

Важно: Семейным парам и студентам с детьми места не предоставляются.

Отдельного внимания заслуживают лабораторные работы по физике. Студенты,
не получившие в течение семестра зачёт по лабораторным работам в
результате наличия одной и более лабораторной работы, которая была не
сделана илик которой студент не был допущен, выполняют её во время
зачётной сессии в установленные лабораторией часы. Если студент не имеет
зачёта по физическому практикуму, к экзамену по физике он не
допускается. Эта информация распространяется и на лабораторные работы по
химии. 

Генерация ответа

In [148]:
response = chat(
    model='qwen3:4b-thinking',
    messages=[
        {
            "role": "system",
            "content": (
                "Ты - помощник студентам МИФИ. "
                "Отвечай только на основе контекста. "
                "Если данных достаточно — ответь кратко и по делу."
                "Если данных нет - напиши 'Информация отсутствует в документе'."
                "Не добавляй фразу об отсутствии информации, если ты уже что-то сказал."
            )
        },
        {
            "role": "user",
            "content": (
                f"Контекст:\n{context}\n\n"
                f"Вопрос: {my_text}"
            )
        }
    ],
)

answer = response.message.content
print(answer)

Информация отсутствует в документе


Метрики

In [149]:
#Подготовим вопросы
# df = pd.read_csv("вопрос_ответ_раг.csv")

# dataset = Dataset.from_dict({
#     "question": df["question"].tolist(),
#     "answer": df["answer"].tolist(),
#     "contexts": df["contexts"].apply(lambda x: [x]).tolist()
# })


# result_metrics = evaluate(
#     dataset,
#     metrics=[faithfulness],
#     llm=ChatOllama(
#         model="llama3:8b-instruct",
#         temperature=0
#     ))